In [ ]:
import os
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field
import pandas as pd
import os

load_dotenv()

c:\Users\Alex\miniconda3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [ ]:
class NewSegmentAPOResponse(BaseModel):
    reasoning: str = Field(description="Проанализируй саму задачу и вопрос в исходном промпте")
    prompt_input_context: str = Field(description="Выделенный входной контекст задачи в одном предложении")
    prompt_task: str = Field(description="Выделенное предложение самой задачи")
    final_prompt: str = Field(description="Финальный промпт")

llm = ChatOpenAI(
    model="openai/gpt-5.4",
    openai_api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
    temperature=0,
    max_retries=10,
    extra_body={
        'guided_json': NewSegmentAPOResponse.model_json_schema()
    }
)
structured_llm = llm.with_structured_output(NewSegmentAPOResponse, method="json_schema")

In [ ]:
df = pd.read_excel('data/test prompts short.xlsx', sheet_name='Targets', header=None)
df.columns = ['idx', 'input', 'something']
print(df.head(10))
df = df[['idx', 'input']]
df

   idx                                              input  \
0    1  Прочитай текст и ответь на вопрос по его содер...   
1    2  Прочитай текст и ответь на вопрос по его содер...   
2    3  Отзыв представляет собой сообщение потребителя...   
3    4  Анализ аннотации позволяет сформировать предст...   
4    5  Прочитай текст и ответь на вопрос по его содер...   
5    6  Бобби и Салли — братья и сестры, которые в нас...   
6    7  Аня и Боря играют в камень-ножницы-бумага. В э...   
7    8  Перед тобой находится утверждение. Проверь его...   
8    9  Прочитай текст и ответь на вопрос по его содер...   
9   10  Есть запрос пользователя: "Я изучаю автомобиль...   

                     something  
0                  в 1629 году  
1              лихеноиндикация  
2                     positive  
3                  лингвистика  
4  низкий уровень безопасности  
5                   увеличится  
6                            9  
7                  Некорректно  
8    Исторической лингвистикой 

,idx,input
0,1,Прочитай текст и ответь на вопрос по его содер...
1,2,Прочитай текст и ответь на вопрос по его содер...
2,3,Отзыв представляет собой сообщение потребителя...
3,4,Анализ аннотации позволяет сформировать предст...
4,5,Прочитай текст и ответь на вопрос по его содер...
5,6,"Бобби и Салли — братья и сестры, которые в нас..."
6,7,Аня и Боря играют в камень-ножницы-бумага. В э...
7,8,Перед тобой находится утверждение. Проверь его...
8,9,Прочитай текст и ответь на вопрос по его содер...
9,10,"Есть запрос пользователя: ""Я изучаю автомобиль..."


In [4]:
SYSTEM_PROMPT = """
## Роль 
Ты экспертный промпт-инженер, который разбирается в оптимизации промптов.

## Задача
Тебе даны: исходный промпт от пользователя (в котором есть входной контекст задачи и вопрос) и эталонный ответ на вопрос. Твоя задача составить сжатый промпт по схеме:
1. Сначала идет входной контекст о задаче в одном предложении в сжатом виже, с наличием информации об эталонном ответе на вопрос.
2. Далее ставится целевой запрос/вопрос, что требуется в данной задаче (в одном предложении)

Задачу решай по шагам:
- Сначала просмотри и проанализируй текст задачи, оцени где в предложении хранится ответ на вопрос, сравнивая с эталонным ответом.
- Составь ОЧЕНЬ СИЛЬНО сжатую версию (не более 10 слов) входного контекста задачи, включив место с потенциальным ответом.
- Составь ОЧЕНЬ СИЛЬНО сжатую версию (не более 10 слов) целевого вопроса/запроса c добавлением "ответь кратко".
- Объедини два предложения в один финальный промпт.

## Формат ответа
- Строго следуй формату JSON
- Ориентируйся только на текст изначального промпта
- За несуществующие слова ты будешь оштрафован
"""
USER_PROMPT = "## Исходный промпт: {}"

In [5]:
import asyncio

async def run_pipeline():
    tasks = [
        structured_llm.ainvoke([
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": USER_PROMPT.format(sample)}
        ]) for sample in df["input"].values
    ]
    
    responses = await asyncio.gather(*tasks)
    
    new_df = pd.DataFrame([{
        "reasoning": r.reasoning,
        "prompt_input_context": r.prompt_input_context,
        "prompt_task": r.prompt_task,
        "final_prompt": r.final_prompt
    } for r in responses])
    
    df["final_prompt"] = new_df["final_prompt"].values
    return df

df = await run_pipeline()

In [6]:
df

,idx,input,final_prompt
0,1,Прочитай текст и ответь на вопрос по его содер...,Джованни Бранка предложил машину в 1629 году. ...
1,2,Прочитай текст и ответь на вопрос по его содер...,Качества воздуха: лихеноиндикация. Как называе...
2,3,Отзыв представляет собой сообщение потребителя...,Отзыв: «РУБАШКА ЛИНЯЕТ!!!» и «Смело РЕКОМЕНДУЮ...
3,4,Анализ аннотации позволяет сформировать предст...,Языковые средства фантастического в политическ...
4,5,Прочитай текст и ответь на вопрос по его содер...,Мотели: к недостаткам относится низкий уровень...
5,6,"Бобби и Салли — братья и сестры, которые в нас...",Бобби живет где регулярно возникают лесные пож...
6,7,Аня и Боря играют в камень-ножницы-бумага. В э...,"36 партий: Аня 6,5,25; Боря 2,31,3; ничьей не ..."
7,8,Перед тобой находится утверждение. Проверь его...,Утверждение о календаре; ответ: «Корректно» ил...
8,9,Прочитай текст и ответь на вопрос по его содер...,Дьяволова нога: увлечение Холмса исторической ...
9,10,"Есть запрос пользователя: ""Я изучаю автомобиль...",Hyundai в 1998 меняла имидж из-за проблемной р...


In [8]:
df.to_csv('data/result_short.csv', index=False)